In [0]:
WITH country_market AS (
    SELECT
        country,

        MAX(CAST(sellers AS DOUBLE)) AS sellers,
        MAX(CAST(buyers AS DOUBLE)) AS buyers,

        MAX(CAST(totalproductslisted AS DOUBLE))
            AS products_listed,

        MAX(CAST(totalproductssold AS DOUBLE))
            AS products_sold,

        MAX(CAST(totalproductswished AS DOUBLE))
            AS products_wished,

        MAX(CAST(totalproductsbought AS DOUBLE))
            AS products_bought,

        MAX(CAST(femalesellersratio AS DOUBLE))
            AS female_seller_ratio,

        MAX(CAST(femalebuyersratio AS DOUBLE))
            AS female_buyer_ratio

    FROM ecommerce_fashion.gold.comprehensive_table
    WHERE sellers IS NOT NULL
       OR buyers IS NOT NULL
    GROUP BY country
),
market_metrics AS (
    SELECT
        *,

        TRY_DIVIDE(buyers, sellers)
            AS buyers_per_seller,

        TRY_DIVIDE(products_wished, products_listed)
            AS wishlist_demand_per_listing,

        100.0 * TRY_DIVIDE(products_sold, products_listed)
            AS sell_through_pct,

        100.0 * TRY_DIVIDE(products_bought, products_wished)
            AS wishlist_conversion_pct

    FROM country_market
)
SELECT
    country,
    CAST(sellers AS BIGINT) AS sellers,
    CAST(buyers AS BIGINT) AS buyers,

    ROUND(buyers_per_seller, 2) AS buyers_per_seller,
    ROUND(wishlist_demand_per_listing, 2)
        AS wishlist_demand_per_listing,

    ROUND(sell_through_pct, 2) AS sell_through_pct,
    ROUND(wishlist_conversion_pct, 2)
        AS wishlist_conversion_pct,

    ROUND(100.0 * female_seller_ratio, 2)
        AS female_seller_pct,

    ROUND(100.0 * female_buyer_ratio, 2)
        AS female_buyer_pct,

    DENSE_RANK() OVER (
        ORDER BY wishlist_demand_per_listing DESC
    ) AS market_opportunity_rank

FROM market_metrics
ORDER BY market_opportunity_rank;